AI Assistance: OpenAI ChatGPT and Anthropic's Claude were used for code debugging, code generation, code organization, and code methodological brainstorming\. All final modeling, implementation, validation, commentary, and interpretation were performed and verified by the authors\.

In [1]:
import pandas as pd
import duckdb
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

platform = 'deepnote'

if platform == 'deepnote':
    PROCESSED_DIR = Path('data/Processed')
if platform == 'vscode':
    PROCESSED_DIR = Path('work/Processed')

# I\. Load real\-time data

In [2]:
# Replace the parquet path below once you have new data
df = pd.read_parquet(PROCESSED_DIR / 'RT_Data.parquet')

# II. Apply log transform

In [3]:
def add_log_transformed_columns(df, columns):
    """Adds log1p-transformed copies of the given column(s) as new '<col>_log' columns."""
    df = df.copy()
    if isinstance(columns, str):
        columns = [columns]

    for col in columns:
        negative_count = (df[col] < 0).sum()
        if negative_count > 0:
            print(f'Warning: {col} has {negative_count:,} negative values; log1p will produce NaN for these.')
        df[f'{col}_log'] = np.log1p(df[col])
        print(f'Added {col}_log — min={df[f"{col}_log"].min():.4f}, max={df[f"{col}_log"].max():.4f}, '
              f'NaNs={df[f"{col}_log"].isna().sum():,}')

    return df

In [4]:
# For real-time data, we log transform the same columns that were done for the historical models
col_list = ['mag_avg_nt', 'flow_speed_km_s', 'proton_density_n_cc']

df_log = add_log_transformed_columns(df=df, columns=col_list)

print(" ")
df_log.info()

Added mag_avg_nt_log — min=1.8886, max=2.0819, NaNs=0
Added flow_speed_km_s_log — min=5.9999, max=6.1614, NaNs=0
Added proton_density_n_cc_log — min=1.2585, max=1.9838, NaNs=0
 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   datetime                 480 non-null    datetime64[ns]
 1   year                     480 non-null    int32         
 2   month                    480 non-null    int32         
 3   day                      480 non-null    int32         
 4   hour                     480 non-null    int32         
 5   minute                   480 non-null    int32         
 6   mag_avg_nt               480 non-null    float64       
 7   bx_gsm_nt                480 non-null    float64       
 8   by_gsm_nt                480 non-null    float64       
 9   bz_gsm_nt                480 non-null    

In [5]:
import matplotlib.pyplot as plt

def plot_histogram(df, column, bins=50):
    """Plots a histogram for the given column and prints summary stats."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(df[column].dropna(), bins=bins, color='#4C72B0', edgecolor='white')
    ax.set_title(f'Distribution of {column}')
    ax.set_xlabel(column)
    ax.set_ylabel('Frequency')
    plt.show()

    print(f'{column}: n={df[column].notna().sum():,}, mean={df[column].mean():.4f}, '
          f'std={df[column].std():.4f}, skew={df[column].skew():.4f}')

In [6]:
# # Plot the distributions of the transformed features
# log_col_list = ['mag_avg_nt_log', 'flow_speed_km_s_log', 'proton_density_n_cc_log']

# for col in log_col_list:
#     plot_histogram(df=df_log, column=col, bins=50)

In [7]:
# Write real-time data with log-transformed columns to parquet
output_path = PROCESSED_DIR / 'RT_data_cleaned_log_transformed.parquet'

df_log.to_parquet(output_path)

print(f'Wrote {len(df_log):,} rows to {output_path.as_posix()}')

Wrote 480 rows to data/Processed/RT_data_cleaned_log_transformed.parquet


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a18bf7d-431c-4909-af8a-a54a62228a78' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>